# 12. Multi-Agent Orchestration (코드 기반 멀티에이전트 오케스트레이션)

**04_Handoffs**에서는 LLM이 자율적으로 핸드오프를 결정하는 **LLM 기반 오케스트레이션**을 배웠습니다.
이번에는 **코드로 직접** 에이전트 흐름을 제어하는 패턴을 학습합니다.

### LLM 기반 vs 코드 기반 오케스트레이션

| 구분 | LLM 기반 (Handoffs) | 코드 기반 |
|------|-------------------|----------|
| 제어 주체 | LLM이 자율 판단 | 개발자가 코드로 제어 |
| 예측 가능성 | 낮음 (LLM 판단에 의존) | 높음 (결정적 흐름) |
| 유연성 | 높음 | 보통 |
| 비용/속도 | 변동적 | 예측 가능 |
| 적합한 경우 | 복잡한 대화, 분류가 어려운 경우 | 파이프라인, 배치 처리, 정해진 워크플로우 |

### 주요 패턴
1. **순차 체이닝 (Sequential Chaining)**: A → B → C
2. **병렬 실행 (Parallelization)**: A, B, C 동시 실행
3. **분류 → 라우팅 (Classification + Routing)**: 분류 후 전문 에이전트에 전달

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import openai
import asyncio

Model = "gpt-5-mini"

## 1. 순차 체이닝 (Sequential Chaining)

에이전트A의 출력을 에이전트B의 입력으로 전달하는 파이프라인입니다.
각 단계가 순서대로 실행됩니다.

```
[작성 에이전트] → 초안 → [편집 에이전트] → 수정본 → [번역 에이전트] → 최종 결과
```

In [3]:
from agents import Agent, Runner, trace

# Step 1: 글 작성 에이전트
writer_agent = Agent(
    name="작성기",
    instructions="""당신은 기술 블로그 작성자입니다.
주어진 주제에 대해 3~4문장의 짧은 기술 소개글을 작성하세요.
전문 용어를 적절히 사용하되 이해하기 쉽게 작성하세요.""",
    model=Model,
)

# Step 2: 편집 에이전트
editor_agent = Agent(
    name="편집기",
    instructions="""당신은 전문 편집자입니다.
주어진 글을 다음 기준으로 수정하세요:
- 문법 오류 수정
- 문장 가독성 향상
- 불필요한 표현 제거
수정된 글만 출력하세요.""",
    model=Model,
)

# Step 3: 번역 에이전트
translator_agent = Agent(
    name="번역기",
    instructions="""당신은 한영 번역가입니다.
주어진 한국어 글을 자연스러운 영어로 번역하세요.
번역된 영어만 출력하세요.""",
    model=Model,
)

# 순차 체이닝 실행
with trace("순차 체이닝: 작성 → 편집 → 번역"):
    # Step 1: 글 작성
    write_result = await Runner.run(writer_agent, "에이전트 AI에 대해 소개글을 작성해주세요.")
    print(f"[Step 1] 초안:\n{write_result.final_output}\n")

    # Step 2: 편집 (이전 단계의 출력을 입력으로 전달)
    edit_result = await Runner.run(editor_agent, write_result.final_output)
    print(f"[Step 2] 편집본:\n{edit_result.final_output}\n")

    # Step 3: 번역 (이전 단계의 출력을 입력으로 전달)
    translate_result = await Runner.run(translator_agent, edit_result.final_output)
    print(f"[Step 3] 번역본:\n{translate_result.final_output}")

[Step 1] 초안:
에이전트 AI는 주어진 목표를 달성하기 위해 스스로 관찰하고 추론하며 행동하는 자율 소프트웨어 구성요소입니다. 최근에는 대형 언어모델(LLM)을 중심으로 자연어로 계획을 세우고 API나 브라우저 같은 도구를 호출해 외부 환경과 상호작용하는 방식이 활발히 연구·실용화되고 있습니다. 에이전트는 메모리(상태 추적), 작업 분해(planning), 반복적 피드백 루프를 통해 복잡한 멀티스텝 작업을 수행하며 고객 지원, 자동화 워크플로우, 연구 보조 등 다양한 분야에 적용됩니다. 다만 안전성, 일관된 추론, 오용 방지, 계산 비용 같은 기술적·윤리적 과제가 남아 있어 신중한 설계와 검증이 필요합니다.

[Step 2] 편집본:
에이전트 AI는 주어진 목표를 달성하기 위해 스스로 관찰·추론·행동하는 자율 소프트웨어 구성 요소입니다. 최근에는 대형 언어 모델(LLM)을 활용해 자연어로 계획을 세우고 API나 브라우저 등 도구를 호출해 외부 환경과 상호작용하는 방식이 활발히 연구·실용화되고 있습니다. 에이전트는 메모리(상태 추적), 작업 분해(계획 수립), 반복적 피드백 루프를 통해 복잡한 다단계 작업을 수행하며 고객 지원, 자동화 워크플로우, 연구 보조 등 다양한 분야에 적용됩니다. 다만 안전성, 일관된 추론, 오용 방지, 계산 비용 등의 기술적·윤리적 과제가 남아 있어 신중한 설계와 검증이 필요합니다.

[Step 3] 번역본:
An AI agent is an autonomous software component that observes, reasons, and acts on its own to achieve a given goal. Recently, using large language models (LLMs) to plan in natural language and invoke tools such as APIs or browsers to interact with external environments has been actively resear

## 2. 병렬 실행 (Parallelization)

서로 독립적인 에이전트를 `asyncio.gather()`로 **동시에** 실행하여 시간을 절약합니다.

```
        ┌→ [감성 분석 에이전트] → 결과1   ─┐
[입력] ──┼→ [키워드 추출 에이전트] → 결과2 ─┼→ [종합]
        └→ [요약 에이전트] → 결과3   ─────┘
```

In [4]:
# 3개의 독립적인 분석 에이전트
sentiment_agent = Agent(
    name="감성 분석기",
    instructions="""주어진 텍스트의 감성을 분석하세요.
결과 형식: "감성: [긍정/부정/중립], 신뢰도: [높음/보통/낮음], 이유: [간단한 설명]" """,
    model=Model,
)

keyword_agent = Agent(
    name="키워드 추출기",
    instructions="""주어진 텍스트에서 핵심 키워드 3~5개를 추출하세요.
결과 형식: "키워드: [키워드1], [키워드2], [키워드3], ..." """,
    model=Model,
)

summary_agent = Agent(
    name="요약기",
    instructions="""주어진 텍스트를 한 문장으로 요약하세요.
결과 형식: "요약: [한 문장 요약]" """,
    model=Model,
)

text_to_analyze = """
OpenAI의 Agent SDK는 개발자들이 AI 에이전트를 쉽게 만들 수 있도록 설계되었습니다.
핸드오프, 도구 사용, 가드레일 등의 기능을 제공하여 복잡한 AI 워크플로우를
간단한 코드로 구현할 수 있게 해줍니다. 특히 Python 개발자에게 친숙한 인터페이스를
제공하며, 프로덕션 환경에서의 안정성도 고려되어 있습니다.
"""

# asyncio.gather로 3개 에이전트를 동시에 실행
with trace("병렬 분석"):
    sentiment_result, keyword_result, summary_result = await asyncio.gather(
        Runner.run(sentiment_agent, text_to_analyze),
        Runner.run(keyword_agent, text_to_analyze),
        Runner.run(summary_agent, text_to_analyze),
    )

print(f"{sentiment_result.final_output}")
print(f"{keyword_result.final_output}")
print(f"{summary_result.final_output}")

감성: 긍정, 신뢰도: 높음, 이유: 제품의 장점(간편성, 다양한 기능 제공, Python 친화성, 프로덕션 안정성)을 긍정적으로 서술하고 있어 호의적인 평가 표현임.
키워드: OpenAI Agent SDK, 핸드오프, 도구 사용, 가드레일, Python 친화적 인터페이스
요약: OpenAI의 Agent SDK는 핸드오프, 도구 사용, 가드레일 등 기능으로 복잡한 AI 워크플로우를 간단한 코드로 구현할 수 있게 하며 Python 친화적 인터페이스와 프로덕션 수준의 안정성을 제공한다.


## 3. 분류 → 라우팅 (Classification + Routing)

**Structured Output**으로 입력을 분류한 후, 결과에 따라 적절한 전문 에이전트를 실행합니다.

```
[입력] → [분류 에이전트] → 카테고리 → [전문 에이전트 선택] → [전문 에이전트 실행] → [응답]
```

In [5]:
from pydantic import BaseModel

# 분류 결과를 위한 Pydantic 모델
class RequestClassification(BaseModel):
    category: str  # "기술지원", "결제문의", "일반문의"
    confidence: float  # 0.0 ~ 1.0
    reason: str

# 분류 에이전트
classifier_agent = Agent(
    name="요청 분류기",
    instructions="""고객의 요청을 다음 카테고리 중 하나로 분류하세요:
- "기술지원": 소프트웨어 오류, 설치 문제, 기술적 질문
- "결제문의": 요금, 환불, 결제 수단, 구독 관련
- "일반문의": 영업시간, 위치, 서비스 소개 등 일반 질문

category에 정확한 카테고리명을 넣고, confidence에 확신도(0~1)를 넣으세요.""",
    model=Model,
    output_type=RequestClassification,
)

# 전문 에이전트들
tech_support_agent = Agent(
    name="기술지원 전문가",
    instructions="당신은 기술지원 전문가입니다. 소프트웨어 문제를 해결하고 기술적 도움을 제공합니다.",
    model=Model,
)

billing_agent = Agent(
    name="결제 전문가",
    instructions="당신은 결제 전문가입니다. 요금, 환불, 구독 관련 질문에 답변합니다.",
    model=Model,
)

general_agent = Agent(
    name="일반 상담사",
    instructions="당신은 일반 상담사입니다. 서비스에 대한 일반적인 질문에 답변합니다.",
    model=Model,
)

# 카테고리 → 에이전트 매핑
agent_map = {
    "기술지원": tech_support_agent,
    "결제문의": billing_agent,
    "일반문의": general_agent,
}

In [6]:
# 분류 → 라우팅 실행
async def handle_customer_request(request: str):
    with trace("분류 → 라우팅"):
        # Step 1: 분류
        classification_result = await Runner.run(classifier_agent, request)
        classification = classification_result.final_output
        print(f"분류 결과: {classification.category} (확신도: {classification.confidence:.0%})")
        print(f"   이유: {classification.reason}")

        # Step 2: 라우팅 - 분류 결과에 따라 적절한 에이전트 선택
        selected_agent = agent_map.get(classification.category, general_agent)
        print(f"선택된 에이전트: {selected_agent.name}\n")

        # Step 3: 전문 에이전트 실행
        response = await Runner.run(selected_agent, request)
        print(f"응답: {response.final_output}")

    return response.final_output

In [7]:
# 테스트 1: 기술지원 질문
print("=" * 60)
print("테스트 1: 기술지원")
print("=" * 60)
await handle_customer_request("프로그램이 갑자기 멈추고 에러 메시지가 떠요. 어떻게 해야 하나요?")

테스트 1: 기술지원
분류 결과: 기술지원 (확신도: 95%)
   이유: 프로그램이 멈추고 에러 메시지가 발생한 것은 소프트웨어 오류 및 문제 해결 관련 문의에 해당하므로 기술지원으로 분류됩니다.
선택된 에이전트: 기술지원 전문가

응답: 먼저 당황하지 마세요. 문제 원인을 좁히기 위해 아래 순서대로 따라 해 보시고, 마지막에 추가로 필요한 정보를 알려주시면 더 구체적으로 도와드릴게요.

1) 기본 확인(간단하고 빠른 해결 가능성)
- 에러 메시지의 정확한 문구(가능하면 복사/스크린샷) 를 저장하세요.
- 프로그램과 컴퓨터/폰을 완전히 재시작해 보세요.
- 프로그램과 운영체제(OS)를 최신 상태로(업데이트) 유지하세요.
- 다른 프로그램을 모두 닫아 메모리·CPU 여유를 확보하고 다시 실행해 보세요.
- 디스크 여유 공간이 충분한지 확인하세요(특히 임시파일 생성이 필요한 경우).

2) 자주 효과적인 조치
- 프로그램을 관리자 권한(윈도우) 또는 sudo 권한(리눅스)으로 실행해 보기.
- 플러그인·확장 기능이 있으면 비활성화 후 실행해 보기.
- 캐시·임시 파일 삭제(프로그램 설정 또는 %TEMP%, /tmp 등).
- 안티바이러스가 차단했는지 확인(임시로 실시간 검사 끄고 테스트).
- 프로그램 삭제 후 최신 버전으로 재설치(설정 백업 권장).

3) 운영체제별 추가 점검
- Windows:
  - 이벤트 뷰어(eventvwr.msc)에서 응용 프로그램/시스템 로그 확인.
  - 시스템 파일 검사: 관리자 명령 프롬프트에서 sfc /scannow, 필요 시 DISM 복구.
  - 드라이버(특히 그래픽) 업데이트.
- macOS:
  - 콘솔(Console)에서 관련 로그 확인.
  - 안전 모드로 부팅(Shift 누르고 부팅) 후 앱 실행 테스트.
- Linux:
  - journalctl -xe 또는 dmesg로 커널·시스템 로그 확인.
  - 라이브러리/종속성 문제인지 ldd로 확인.

4) 더 깊은 진단(계속 재현되면)
- 문제 재현 절차를 정리: 

'먼저 당황하지 마세요. 문제 원인을 좁히기 위해 아래 순서대로 따라 해 보시고, 마지막에 추가로 필요한 정보를 알려주시면 더 구체적으로 도와드릴게요.\n\n1) 기본 확인(간단하고 빠른 해결 가능성)\n- 에러 메시지의 정확한 문구(가능하면 복사/스크린샷) 를 저장하세요.\n- 프로그램과 컴퓨터/폰을 완전히 재시작해 보세요.\n- 프로그램과 운영체제(OS)를 최신 상태로(업데이트) 유지하세요.\n- 다른 프로그램을 모두 닫아 메모리·CPU 여유를 확보하고 다시 실행해 보세요.\n- 디스크 여유 공간이 충분한지 확인하세요(특히 임시파일 생성이 필요한 경우).\n\n2) 자주 효과적인 조치\n- 프로그램을 관리자 권한(윈도우) 또는 sudo 권한(리눅스)으로 실행해 보기.\n- 플러그인·확장 기능이 있으면 비활성화 후 실행해 보기.\n- 캐시·임시 파일 삭제(프로그램 설정 또는 %TEMP%, /tmp 등).\n- 안티바이러스가 차단했는지 확인(임시로 실시간 검사 끄고 테스트).\n- 프로그램 삭제 후 최신 버전으로 재설치(설정 백업 권장).\n\n3) 운영체제별 추가 점검\n- Windows:\n  - 이벤트 뷰어(eventvwr.msc)에서 응용 프로그램/시스템 로그 확인.\n  - 시스템 파일 검사: 관리자 명령 프롬프트에서 sfc /scannow, 필요 시 DISM 복구.\n  - 드라이버(특히 그래픽) 업데이트.\n- macOS:\n  - 콘솔(Console)에서 관련 로그 확인.\n  - 안전 모드로 부팅(Shift 누르고 부팅) 후 앱 실행 테스트.\n- Linux:\n  - journalctl -xe 또는 dmesg로 커널·시스템 로그 확인.\n  - 라이브러리/종속성 문제인지 ldd로 확인.\n\n4) 더 깊은 진단(계속 재현되면)\n- 문제 재현 절차를 정리: 어떤 버튼/작업에서 멈추는지 단계별로 적기.\n- 에러 코드, 스택트레이스, 로그 파일(프로그램 로그 위치 확인) 수집.\n- 프로그램이 크래시 덤프를 생성하면(dump 파일) 저장해서 제

In [8]:
# 테스트 2: 결제 질문
print("=" * 60)
print("테스트 2: 결제문의")
print("=" * 60)
await handle_customer_request("지난달 결제가 두 번 된 것 같아요. 환불 가능한가요?")

테스트 2: 결제문의
분류 결과: 결제문의 (확신도: 98%)
   이유: 사용자가 '지난달 결제가 두 번 된 것 같다'며 '환불 가능한가요?'라고 문의함 — 결제 내역 및 환불 관련 질문으로 결제문의에 해당함.
선택된 에이전트: 결제 전문가

응답: 대응 가능합니다. 보통 중복 결제는 환불 가능한 경우가 많으니 다음 정보를 확인해 주세요 — 제가 바로 도와드릴 수 있습니다.

1) 먼저 확인해 주세요
- 결제 수단(신용카드, 체크카드, 계좌이체, 휴대폰결제, App Store/Google Play 등)
- 결제일자와 금액(은행·카드 명세서에 찍힌 날짜/금액)
- 가맹점명(명세서에 뜨는 청구명)
- 거래ID/승인번호나 영수증(있으면 스크린샷)

2) 바로 할 일
- 은행/카드사 앱이나 인터넷뱅킹에서 같은 금액이 2건(posted)인지, 하나는 승인보류(authorization)인지 확인
  - 승인보류는 실제 이중인출이 아닐 수 있고 며칠 내에 자동으로 사라집니다.
- 가맹점(판매자) 고객센터에 연락해서 “중복 결제로 보이니 환불(또는 취소) 요청한다”라고 요청
  - 가맹점이 환불 처리해야 카드사로 환불이 이뤄집니다.

3) 가맹점에 보낼 간단한 요청문(복사해서 쓰세요)
안녕하세요. 고객님 성함(또는 회원ID)입니다.  
[결제일]에 [금액]원이 결제되었는데, 동일한 금액이 동일 일자에 두 건 발생한 것으로 확인됩니다(거래ID: xxxx). 중복결제로 판단되어 중복분 환불 요청드립니다. 확인 후 처리 결과 회신 부탁드립니다. 감사합니다.

4) 가맹점 응답이 없거나 거절하면
- 신용/체크카드: 카드사에 ‘이의거래(부당청구) 신청’ → 카드사가 조사 후 환불(차지백) 진행 가능
  - 카드사 접수 기한은 통상 구매일로부터 60~120일(카드사별 상이), 처리기간 보통 7~30영업일
- 계좌이체/가상계좌: 은행·결제대행사에 이의제기 → 처리기간 상이
- App Store/Google Play: 각사 환불절차(앱스토어·구글플레이 고객센터나 구매내역에서 환불 요청)

'대응 가능합니다. 보통 중복 결제는 환불 가능한 경우가 많으니 다음 정보를 확인해 주세요 — 제가 바로 도와드릴 수 있습니다.\n\n1) 먼저 확인해 주세요\n- 결제 수단(신용카드, 체크카드, 계좌이체, 휴대폰결제, App Store/Google Play 등)\n- 결제일자와 금액(은행·카드 명세서에 찍힌 날짜/금액)\n- 가맹점명(명세서에 뜨는 청구명)\n- 거래ID/승인번호나 영수증(있으면 스크린샷)\n\n2) 바로 할 일\n- 은행/카드사 앱이나 인터넷뱅킹에서 같은 금액이 2건(posted)인지, 하나는 승인보류(authorization)인지 확인\n  - 승인보류는 실제 이중인출이 아닐 수 있고 며칠 내에 자동으로 사라집니다.\n- 가맹점(판매자) 고객센터에 연락해서 “중복 결제로 보이니 환불(또는 취소) 요청한다”라고 요청\n  - 가맹점이 환불 처리해야 카드사로 환불이 이뤄집니다.\n\n3) 가맹점에 보낼 간단한 요청문(복사해서 쓰세요)\n안녕하세요. 고객님 성함(또는 회원ID)입니다.  \n[결제일]에 [금액]원이 결제되었는데, 동일한 금액이 동일 일자에 두 건 발생한 것으로 확인됩니다(거래ID: xxxx). 중복결제로 판단되어 중복분 환불 요청드립니다. 확인 후 처리 결과 회신 부탁드립니다. 감사합니다.\n\n4) 가맹점 응답이 없거나 거절하면\n- 신용/체크카드: 카드사에 ‘이의거래(부당청구) 신청’ → 카드사가 조사 후 환불(차지백) 진행 가능\n  - 카드사 접수 기한은 통상 구매일로부터 60~120일(카드사별 상이), 처리기간 보통 7~30영업일\n- 계좌이체/가상계좌: 은행·결제대행사에 이의제기 → 처리기간 상이\n- App Store/Google Play: 각사 환불절차(앱스토어·구글플레이 고객센터나 구매내역에서 환불 요청)\n\n5) 환불 처리 시간 및 유의사항\n- 카드 환불: 카드사·가맹점 처리 후 실제 환불액이 계좌에 들어오기까지 보통 영업일 기준 3–14일, 최대 한 달 가량 걸릴 수 있음\n- 승인보류(authorizat

In [9]:
# 테스트 3: 일반 질문
print("=" * 60)
print("테스트 3: 일반문의")
print("=" * 60)
await handle_customer_request("주말에도 고객센터가 운영되나요?")

테스트 3: 일반문의
분류 결과: 일반문의 (확신도: 97%)
   이유: 고객센터 운영 시간(주말 운영 여부)을 묻는 질문으로, 영업시간 등 일반적 정보에 해당하므로 '일반문의'로 분류했습니다.
선택된 에이전트: 일반 상담사

응답: 회사마다 다릅니다.  
- 일부는 주말에도 평일과 동일하게 운영하거나 단축(예: 토요일 오전만)하고,  
- 일부는 주말·공휴일에 휴무이며 대신 FAQ·챗봇·앱·1:1문의는 이용 가능하게 해 놓습니다.

확인 방법: 회사 홈페이지의 고객센터/운영시간 안내, 앱 내 고객지원, 대표전화 자동응답 메시지나 SNS 공지 등을 확인하세요. 원하시면 회사(또는 서비스) 이름 알려주시면 운영시간을 대신 확인해 드릴게요.


'회사마다 다릅니다.  \n- 일부는 주말에도 평일과 동일하게 운영하거나 단축(예: 토요일 오전만)하고,  \n- 일부는 주말·공휴일에 휴무이며 대신 FAQ·챗봇·앱·1:1문의는 이용 가능하게 해 놓습니다.\n\n확인 방법: 회사 홈페이지의 고객센터/운영시간 안내, 앱 내 고객지원, 대표전화 자동응답 메시지나 SNS 공지 등을 확인하세요. 원하시면 회사(또는 서비스) 이름 알려주시면 운영시간을 대신 확인해 드릴게요.'

## 4. 종합 패턴: 병렬 + 순차 + 체이닝 조합

실전에서는 여러 패턴을 **조합**하여 사용합니다.

```
[고객 리뷰] ──→ [병렬 분석] ──→ [종합 에이전트] ──→ [최종 보고서]
                 ├ 감성 분석
                 ├ 키워드 추출
                 └ 요약
```

In [10]:
# 종합 보고서를 작성하는 에이전트
report_agent = Agent(
    name="보고서 작성기",
    instructions="""당신은 분석 보고서 작성자입니다.
주어진 분석 결과들을 종합하여 간결한 보고서를 작성하세요.
보고서 형식:
---
[분석 보고서]
- 요약: ...
- 감성: ...
- 핵심 키워드: ...
- 종합 의견: (위 분석을 바탕으로 한 줄 의견)
---""",
    model=Model,
)

customer_review = """
이 제품 정말 좋아요! 배송도 빠르고 포장도 꼼꼼했습니다.
다만 설명서가 영어로만 되어 있어서 처음에 좀 헷갈렸어요.
전체적으로는 가격 대비 성능이 훌륭하고, 다음에도 이 브랜드를 구매할 의향이 있습니다.
"""

with trace("종합 분석 파이프라인"):
    # Step 1: 병렬 분석 (동시 실행)
    print("Step 1: 병렬 분석 실행 중...\n")
    s_result, k_result, sm_result = await asyncio.gather(
        Runner.run(sentiment_agent, customer_review),
        Runner.run(keyword_agent, customer_review),
        Runner.run(summary_agent, customer_review),
    )

    print(f"  감성: {s_result.final_output}")
    print(f"  키워드: {k_result.final_output}")
    print(f"  요약: {sm_result.final_output}")

    # Step 2: 순차 체이닝 - 분석 결과를 종합
    print(f"\nStep 2: 종합 보고서 작성 중...\n")
    combined_input = f"""다음 분석 결과를 종합해주세요:

원본 리뷰: {customer_review}

분석 결과:
1. {s_result.final_output}
2. {k_result.final_output}
3. {sm_result.final_output}
"""

report_result = await Runner.run(report_agent, combined_input)
print(f"📊 최종 보고서:\n{report_result.final_output}")

Step 1: 병렬 분석 실행 중...

  감성: 감성: 긍정, 신뢰도: 높음, 이유: 제품·배송·포장에 대해 긍정적으로 평가하고 가격 대비 성능이 훌륭하며 재구매 의사까지 밝혀 전체적으로 호의적이지만 설명서 영어 표기로 인한 사소한 불만이 있음.
  키워드: 키워드: 가성비, 빠른 배송, 꼼꼼한 포장, 영문 설명서, 재구매 의향
  요약: 요약: 제품과 배송·포장에 만족했으나 설명서가 영어로만 되어 있어 초반에 혼란이 있었고, 전반적으로 가격 대비 성능이 훌륭해 향후 재구매 의사가 있습니다.

Step 2: 종합 보고서 작성 중...

📊 최종 보고서:
[분석 보고서]
- 요약: 제품과 배송·포장에 대해 매우 만족하며 가격 대비 성능이 훌륭해 재구매 의사가 있으나, 설명서가 영어로만 되어 있어 초반에 다소 혼란이 있었음.
- 감성: 긍정 (신뢰도: 높음) — 전반적으로 호의적이며 재구매 의사까지 밝혀 신뢰도가 높음. 다만 영문 설명서에 대한 사소한 불만 존재.
- 핵심 키워드: 가성비, 빠른 배송, 꼼꼼한 포장, 영문 설명서, 재구매 의향
- 종합 의견: 전반적으로 만족도가 높아 긍정적 평가이나, 한글 설명서 제공으로 사용자 경험을 더 개선하면 재구매 확률이 더 상승할 것 같습니다.


---------------------
### 정리

| 패턴 | 코드 | 적합한 경우 |
|------|------|-----------|
| 순차 체이닝 | `result1 = run(A, input)` → `result2 = run(B, result1.final_output)` | 파이프라인, 단계별 처리 |
| 병렬 실행 | `await asyncio.gather(run(A, x), run(B, x), run(C, x))` | 독립적 분석, 시간 절약 |
| 분류 → 라우팅 | `Structured Output`으로 분류 → `agent_map[category]` | 고객 문의 분류, 의도 분석 |
| 조합 패턴 | 병렬 + 순차를 결합 | 실전 워크플로우 |

**핵심 포인트:**
- `trace()`로 전체 워크플로우를 하나의 트레이스로 묶어 추적
- `asyncio.gather()`로 독립적인 작업을 병렬 실행하여 성능 향상
- `Structured Output (output_type)`으로 결정적(deterministic) 라우팅 구현
- 코드 기반 오케스트레이션은 **예측 가능성**과 **비용 제어**에 유리